# 2/4: Data preprocessing 1
By Niloufar Shahdoust (niloufar.shahdoust@utah.edu)

# pay attention, here it says nmm atlas, but it's actually brainetome here, I've changed it in code 1


In [1]:
import os
import mat73
import numpy as np
import pandas as pd
from matplotlib import cm
from ast import literal_eval
import re
import ast
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from visbrain.objects import BrainObj, SceneObj, SourceObj 

In [2]:
input_folder = '1_brain_visualization_data_retrieval'
output_folder = '2_brain_visualization_preProcessing_1'

os.makedirs(output_folder, exist_ok=True)

csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

for file_name in csv_files:

    file_path = os.path.join(input_folder, file_name)
    df = pd.read_csv(file_path)

    # Process Nmm_atlas
    if 'Nmm_atlas' in df.columns:

        df['Nmm_atlas'] = df['Nmm_atlas'].astype(str).str.strip()

        # Remove unwanted regions
        df = df[
            ~df['Nmm_atlas'].str.contains(
                'Cerebral White Matter|Unknown',
                case=False,
                na=False
            )
        ].copy()

        # Remove brackets and quotation marks
        df['Nmm_atlas'] = (
            df['Nmm_atlas']
            .str.replace(r"^\[\s*['\"]?", '', regex=True)
            .str.replace(r"['\"]?\s*\]$", '', regex=True)
            .str.strip()
        )

        def format_nmm_atlas(atlas_name):

            # Remove an existing Left/Right prefix to prevent duplication
            atlas_name = re.sub(
                r'^(Left|Right)\s+',
                '',
                atlas_name,
                flags=re.IGNORECASE
            ).strip()

            # Amyg_L_2_1 → Left Amyg
            # Amyg_R_2_1 → Right Amyg
            match = re.match(
                r'^(.*)_([LR])(?:_\d+)*$',
                atlas_name,
                flags=re.IGNORECASE
            )

            if match:
                area_name = match.group(1).strip()
                hemisphere = match.group(2).upper()

                if hemisphere == 'L':
                    return f'Left {area_name}'

                return f'Right {area_name}'

            return atlas_name

        # Change the Nmm_atlas column itself
        df['Nmm_atlas'] = df['Nmm_atlas'].apply(format_nmm_atlas)

        # Add hemisphere indicator columns
        df['left'] = (
            df['Nmm_atlas']
            .str.match(r'^Left\s+', case=False, na=False)
            .astype(int)
        )

        df['right'] = (
            df['Nmm_atlas']
            .str.match(r'^Right\s+', case=False, na=False)
            .astype(int)
        )

        # Keep area without Left or Right
        df['area'] = (
            df['Nmm_atlas']
            .str.replace(
                r'^(Left|Right)\s+',
                '',
                regex=True,
                case=False
            )
            .str.strip()
        )

    # Process MNI coordinates
    if 'MNI_Coordinates' in df.columns:

        def parse_coordinates(coord):
            try:
                parsed = ast.literal_eval(str(coord))

                if not isinstance(parsed, (list, tuple)) or len(parsed) != 3:
                    return None

                parsed = [float(value) for value in parsed]

                if any(np.isnan(value) for value in parsed):
                    return None

                return parsed

            except (ValueError, SyntaxError, TypeError):
                return None

        df['parsed_coordinates'] = (
            df['MNI_Coordinates'].apply(parse_coordinates)
        )

        df = df.dropna(subset=['parsed_coordinates']).copy()

        df['coordinate_x'] = df['parsed_coordinates'].str[0]
        df['coordinate_y'] = df['parsed_coordinates'].str[1]
        df['coordinate_z'] = df['parsed_coordinates'].str[2]

        df = df.drop(
            columns=['MNI_Coordinates', 'parsed_coordinates']
        )

    output_path = os.path.join(output_folder, file_name)
    df.to_csv(output_path, index=False)

    print(f'Saved: {output_path}')

os.listdir(output_folder)

Saved: 2_brain_visualization_preProcessing_1\201810.csv
Saved: 2_brain_visualization_preProcessing_1\201811.csv
Saved: 2_brain_visualization_preProcessing_1\201901.csv
Saved: 2_brain_visualization_preProcessing_1\201902.csv
Saved: 2_brain_visualization_preProcessing_1\201903.csv
Saved: 2_brain_visualization_preProcessing_1\201905.csv
Saved: 2_brain_visualization_preProcessing_1\201909.csv
Saved: 2_brain_visualization_preProcessing_1\201910.csv
Saved: 2_brain_visualization_preProcessing_1\201911.csv
Saved: 2_brain_visualization_preProcessing_1\201914.csv
Saved: 2_brain_visualization_preProcessing_1\201915.csv
Saved: 2_brain_visualization_preProcessing_1\202001.csv
Saved: 2_brain_visualization_preProcessing_1\202002.csv
Saved: 2_brain_visualization_preProcessing_1\202003.csv
Saved: 2_brain_visualization_preProcessing_1\202004.csv
Saved: 2_brain_visualization_preProcessing_1\202005.csv
Saved: 2_brain_visualization_preProcessing_1\202006u.csv
Saved: 2_brain_visualization_preProcessing_1\20

['201810.csv',
 '201811.csv',
 '201901.csv',
 '201902.csv',
 '201903.csv',
 '201905.csv',
 '201909.csv',
 '201910.csv',
 '201911.csv',
 '201914.csv',
 '201915.csv',
 '202001.csv',
 '202002.csv',
 '202003.csv',
 '202004.csv',
 '202005.csv',
 '202006u.csv',
 '202007.csv',
 '202008.csv',
 '202011.csv',
 '202014.csv',
 '202015.csv',
 '202016.csv',
 '202105.csv',
 '202107.csv',
 '202110.csv',
 '202114.csv',
 '202117.csv',
 '202118.csv',
 '202201.csv',
 '202202.csv',
 '202205.csv',
 '202207.csv',
 '202208.csv',
 '202212.csv',
 '202215.csv',
 '202216.csv',
 '202217.csv',
 '202302.csv',
 '202306.csv',
 '202307.csv',
 '202308.csv',
 '202309.csv',
 '202311.csv',
 '202314a.csv',
 '202401.csv',
 '202404.csv',
 '202405.csv',
 '202407.csv',
 '202408.csv',
 '202409.csv',
 '202413a.csv',
 '202414.csv',
 '202416.csv',
 '202417.csv',
 '202418.csv',
 '202421.csv',
 '202422.csv',
 '202501.csv',
 '202503.csv',
 '202504.csv',
 '202505.csv',
 '202507.csv',
 '202508.csv',
 '202510.csv',
 '202511.csv',
 '20251